## Instalando bibliotecas

In [4]:
!pip install transformers datasets peft accelerate torch

## Importando bibliotecas

In [5]:
from datasets import load_dataset
from transformers import (
    AutoModelForSeq2SeqLM,
    AutoTokenizer,
    Seq2SeqTrainer,
    Seq2SeqTrainingArguments,
    DataCollatorForSeq2Seq
)
from peft import (
    LoraConfig,
    get_peft_model,
    prepare_model_for_kbit_training
)
import torch

## Preparando Dataset

In [ ]:
def convert_to_hf_format(example):
    """Combina instrução e saída em um único texto."""
    return {
        "text": f"Instruction: {example['Instruction']}\nOutput: {example['Output']}"
    }

# Carrega o arquivo JSON Lines
dataset = load_dataset('json', data_files='../dataset.jsonl')
# Aplica a conversão
dataset = dataset.map(convert_to_hf_format)
# Divide em treino e teste
dataset = dataset["train"].train_test_split(test_size=0.2)
print(dataset)

Generating train split: 0 examples [00:00, ? examples/s]

Map:   0%|          | 0/100 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['Instruction', 'Output', 'text'],
        num_rows: 80
    })
    test: Dataset({
        features: ['Instruction', 'Output', 'text'],
        num_rows: 20
    })
})


## Carregar o Modelos Pré-Treinado e o Tokenizador

In [7]:
model_name = "google/mt5-base"
tokenizer = AutoTokenizer.from_pretrained(model_name)
base_model = AutoModelForSeq2SeqLM.from_pretrained(model_name,torch_dtype=torch.float16)

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

print(f"Modelo carregado: {model_name}")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/702 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/376 [00:00<?, ?B/s]

spiece.model:   0%|          | 0.00/4.31M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/65.0 [00:00<?, ?B/s]

...

Loading weights:   0%|          | 0/284 [00:00<?, ?it/s]

model.safetensors:   0%|          | 0.00/2.33G [00:00<?, ?B/s]

[transformers] The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.


generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

Modelo carregado: google/mt5-base


## Testando Modelo Crú

In [8]:
def generate_response(model, tokenizer, instruction, input_text=""):
    """Gera uma resposta a partir de uma instrução, usando o modelo fornecido."""
    if input_text:
        prompt = f"Instruction: {instruction}\nInput: {input_text}\nOutput:"
    else:
        prompt = f"Instruction: {instruction}\nOutput:"

    inputs = tokenizer(prompt, return_tensors="pt")
    outputs = model.generate(
        input_ids=inputs["input_ids"],
        attention_mask=inputs["attention_mask"],
        max_new_tokens=150,
        pad_token_id=tokenizer.eos_token_id,
        do_sample=True,          # ativa amostragem para usar temperatura
        temperature=0.7
    )
    full_output = tokenizer.decode(outputs[0], skip_special_tokens=True)
    # Extrai apenas a parte após "Output:"
    resposta = full_output.split("Output:")[-1].strip()
    return resposta

# Exemplo de instrução (deve existir no dataset.jsonl)
test_instruction = "Qual é a prioridade ao inspecionar o local de trabalho antes de usar a máquina?"

print("=== ANTES DO FINE-TUNING ===")
print(f"Instrução: {test_instruction}")
print(f"Resposta base: {generate_response(base_model, tokenizer, test_instruction)}")

=== ANTES DO FINE-TUNING ===
Instrução: Qual é a prioridade ao inspecionar o local de trabalho antes de usar a máquina?
Resposta base: <extra_id_0>. Ex.


## Tokenização do Dataset

In [9]:
def tokenize_function(examples):
    model_inputs = tokenizer(
        examples["Instruction"],
        max_length=256,
        truncation=True,
        padding="max_length"
    )

    labels = tokenizer(
        examples["Output"],
        max_length=256,
        truncation=True,
        padding="max_length"
    )

    model_inputs["labels"] = labels["input_ids"]

    return model_inputs

tokenized_datasets = dataset.map(tokenize_function, batched=True)
print("Dataset tokenizado:", tokenized_datasets)

Map:   0%|          | 0/80 [00:00<?, ? examples/s]

Map:   0%|          | 0/20 [00:00<?, ? examples/s]

Dataset tokenizado: DatasetDict({
    train: Dataset({
        features: ['Instruction', 'Output', 'text', 'input_ids', 'attention_mask', 'labels'],
        num_rows: 80
    })
    test: Dataset({
        features: ['Instruction', 'Output', 'text', 'input_ids', 'attention_mask', 'labels'],
        num_rows: 20
    })
})


## Preparar o Modelo para LoRA

In [10]:
# A partir daqui, usaremos uma cópia do modelo base para o fine-tuning
model = base_model  # poderia também ser uma nova instância
model = prepare_model_for_kbit_training(model)

## Verificando Camadas Target

In [11]:
import torch.nn as nn

for name, module in base_model.named_modules():
    if isinstance(module, nn.Linear):
        print(name)

encoder.block.0.layer.0.SelfAttention.q
encoder.block.0.layer.0.SelfAttention.k
encoder.block.0.layer.0.SelfAttention.v
encoder.block.0.layer.0.SelfAttention.o
encoder.block.0.layer.1.DenseReluDense.wi_0
encoder.block.0.layer.1.DenseReluDense.wi_1
encoder.block.0.layer.1.DenseReluDense.wo
encoder.block.1.layer.0.SelfAttention.q
encoder.block.1.layer.0.SelfAttention.k
encoder.block.1.layer.0.SelfAttention.v
encoder.block.1.layer.0.SelfAttention.o
encoder.block.1.layer.1.DenseReluDense.wi_0
encoder.block.1.layer.1.DenseReluDense.wi_1
encoder.block.1.layer.1.DenseReluDense.wo
encoder.block.2.layer.0.SelfAttention.q
encoder.block.2.layer.0.SelfAttention.k
encoder.block.2.layer.0.SelfAttention.v
encoder.block.2.layer.0.SelfAttention.o
encoder.block.2.layer.1.DenseReluDense.wi_0
encoder.block.2.layer.1.DenseReluDense.wi_1
encoder.block.2.layer.1.DenseReluDense.wo
encoder.block.3.layer.0.SelfAttention.q
encoder.block.3.layer.0.SelfAttention.k
encoder.block.3.layer.0.SelfAttention.v
encoder.bl

## Configurar e Injetar LoRA

In [12]:
!pip install --upgrade torchao

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 50.1 MB/s eta 0:00:00
  Attempting uninstall: torchao
    Found existing installation: torchao 0.10.0
    Uninstalling torchao-0.10.0:
      Successfully uninstalled torchao-0.10.0


In [13]:
lora_config = LoraConfig(
    r=16,                    # rank da decomposição
    lora_alpha=32,           # escala alpha
    target_modules=["q","k","v","o","wi","wo"],  # camadas alvo
    lora_dropout=0.05,
    bias="none",
    task_type="SEQ_2_SEQ_LM",
    inference_mode=False     # False = modo treinamento
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

trainable params: 4,620,288 || all params: 587,021,568 || trainable%: 0.7871


## Data Collator (Modelagem Sequencial)

In [14]:
data_collator = DataCollatorForSeq2Seq(
    tokenizer=tokenizer,
    model=base_model
)

## Argumentos de Treinamento (Preparando Hiperparâmetros)

In [ ]:
training_args = Seq2SeqTrainingArguments(
    output_dir="/modelo_treinado_google",
    learning_rate=2e-4,
    per_device_train_batch_size=4,
    per_device_eval_batch_size=4,
    num_train_epochs=20,
    eval_strategy="steps",
    eval_steps=100,
    save_strategy="steps",
    save_steps=100,
    logging_steps=10,
    predict_with_generate=True,
    report_to="none"
)

## Iniciar Treino

In [16]:
trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_datasets["train"],
    eval_dataset=tokenized_datasets["test"],
    data_collator=data_collator
)

## Treinar Modelo

In [17]:
trainer.train()

Step,Training Loss,Validation Loss
100,13.834764,9.407145
200,3.846275,2.839433
300,1.489866,1.010690
400,1.146948,1.014648


TrainOutput(global_step=400, training_loss=10.532978084087372, metrics={'train_runtime': 318.2518, 'train_samples_per_second': 5.027, 'train_steps_per_second': 1.257, 'total_flos': 970591725158400.0, 'train_loss': 10.532978084087372, 'epoch': 20.0})

## Salvar o Modelo Ajustado e o Tokenizador

In [ ]:
model.save_pretrained("/lora_google_sequencial_finetuned_model")
tokenizer.save_pretrained("/google_tokenizer")

('/content/drive/MyDrive/Colab Notebooks/google_tokenizer/tokenizer_config.json',
 '/content/drive/MyDrive/Colab Notebooks/google_tokenizer/tokenizer.json')

## Teste pós fine-tuning

In [ ]:

# Carrega o modelo fine-tunado (apenas os adaptadores LoRA)
finetuned_model = AutoModelForSeq2SeqLM.from_pretrained("/lora_google_sequencial_finetuned_model")
finetuned_tokenizer = AutoTokenizer.from_pretrained("/google_tokenizer")


# Garante que o token de padding esteja definido
if finetuned_tokenizer.pad_token is None:
    finetuned_tokenizer.pad_token = finetuned_tokenizer.eos_token

Loading weights:   0%|          | 0/284 [00:00<?, ?it/s]

[transformers] The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.


Loading weights:   0%|          | 0/336 [00:00<?, ?it/s]

In [20]:
print("=== DEPOIS DO FINE-TUNING ===")
print(f"Instrução: {test_instruction}")
print(f"Resposta ajustada: {generate_response(finetuned_model, finetuned_tokenizer, test_instruction)}")

=== DEPOIS DO FINE-TUNING ===
Instrução: Qual é a prioridade ao inspecionar o local de trabalho antes de usar a máquina?
Resposta ajustada: <extra_id_0> o ativo de .s,,. de <extra_id_14> deecom doa ap de, am.,,, <extra_id_16> de de,,s deo, <extra_id_14>o de de,.us da, de de de, be  de de dee, de,e das,e ,. aa aa., ,e.  de , de ,, de , de, de dea, dea am, dea de de  de de  a,,, .,  e de . desaa,, de,,,  ,    de,, am ,..,. , de,
